In [45]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import tensorflow as t
from tensorflow.keras.models import Model, load_model
from tensorflow.keras.layers import Input, LSTM, Dense, RepeatVector, TimeDistributed, Dropout, GaussianNoise
from tensorflow.keras.callbacks import EarlyStopping, ReduceLROnPlateau, ModelCheckpoint
from tensorflow.keras.optimizers import Adam
from tensorflow.keras.regularizers import l2
from sklearn.preprocessing import MinMaxScaler
from sklearn.model_selection import train_test_split
import json
import os
import glob
import pickle
from datetime import datetime
from scipy import stats

# Categorical encoder class to ensure consistent encoding across datasets
class CategoricalEncoder:
    def __init__(self):
        self.category_maps = {}
        self.reverse_maps = {}
        
    def fit(self, data, column):
        """Fit encoder on a categorical column"""
        unique_values = sorted(data[column].unique())
        self.category_maps[column] = {val: i+1 for i, val in enumerate(unique_values)}
        # Add a default value for unseen categories
        self.category_maps[column]['__unseen__'] = 0
        # Create reverse mapping for interpretability
        self.reverse_maps[column] = {i+1: val for i, val in enumerate(unique_values)}
        self.reverse_maps[column][0] = '__unseen__'
        
    def transform(self, data, column):
        """Transform categorical values to numerical using consistent mapping"""
        if column not in self.category_maps:
            raise ValueError(f"Encoder not fitted for column: {column}")
            
        # Map values, using 0 for unseen values
        return data[column].map(lambda x: self.category_maps[column].get(x, 0))
    
    def fit_transform(self, data, column):
        """Fit and transform in one step"""
        self.fit(data, column)
        return self.transform(data, column)
    
    def inverse_transform(self, encoded_values, column):
        """Convert encoded values back to original categories"""
        if column not in self.reverse_maps:
            raise ValueError(f"Encoder not fitted for column: {column}")
            
        # Map values back to categories
        if isinstance(encoded_values, pd.Series):
            return encoded_values.map(lambda x: self.reverse_maps[column].get(x, '__unknown__'))
        else:
            return [self.reverse_maps[column].get(x, '__unknown__') for x in encoded_values]
            
    def save(self, filepath):
        """Save encoders to file"""
        with open(filepath, 'wb') as f:
            pickle.dump({'category_maps': self.category_maps, 'reverse_maps': self.reverse_maps}, f)
            
    def load(self, filepath):
        """Load encoders from file"""
        with open(filepath, 'rb') as f:
            data = pickle.load(f)
            self.category_maps = data['category_maps']
            self.reverse_maps = data['reverse_maps']

# JSON dosyalarından akış verilerini yükleme fonksiyonu
def load_flows_from_directory(directory):
    flows = []
    json_files = glob.glob(os.path.join(directory, '*.json'))
    print(f"{directory} klasöründe {len(json_files)} JSON dosyası bulundu.")
    
    for json_file in json_files:
        try:
            with open(json_file, 'r') as f:
                data = json.load(f)
                if 'flows' in data:
                    flows.extend(data['flows'])
                else:
                    print(f"Uyarı: {json_file} dosyasında 'flows' anahtarı bulunamadı.")
        except Exception as e:
            print(f"Hata: {json_file} dosyası yüklenirken bir sorun oluştu: {str(e)}")
    
    print(f"{directory} klasöründen toplam {len(flows)} akış yüklendi.")
    return flows

# Gelişmiş veri hazırlama fonksiyonu
def prepare_flow_data(flows, encoders=None, is_training=True, reference_stats=None):
    """
    Enhanced flow data preparation with consistent encoding, feature engineering, and reference-based normalization
    while maintaining backward compatibility
    
    Parameters:
    -----------
    flows : list
        List of flow dictionaries
    encoders : CategoricalEncoder or None
        Encoders for categorical features (created during training)
    is_training : bool
        Whether this is training data or not
    reference_stats : dict or None
        Statistics from training data for consistent normalization
        
    Returns:
    --------
    df : pandas.DataFrame
        Prepared DataFrame
    encoders : CategoricalEncoder
        Updated encoders object
    reference_stats : dict (only if reference_stats parameter is provided)
        Dictionary containing statistical references for each feature
    """
    # Convert to DataFrame
    df = pd.DataFrame(flows)
    
    # Basic preprocessing
    df['start_time'] = pd.to_datetime(df['start_time'])
    df['end_time'] = pd.to_datetime(df['end_time'])
    
    # Handle missing or invalid values
    # Check for NaN values
    nan_count = df.isna().sum()
    if nan_count.sum() > 0:
        print(f"NaN değerleri tespit edildi:\n{nan_count[nan_count > 0]}")
        # Fill NaN values appropriately
        for col in df.columns:
            if df[col].dtype in ['int64', 'float64']:
                df[col].fillna(0, inplace=True)
            else:
                df[col].fillna('unknown', inplace=True)
    
    # Check for negative durations
    if 'duration_seconds' in df.columns and (df['duration_seconds'] < 0).any():
        print(f"Uyarı: {(df['duration_seconds'] < 0).sum()} akışın süresi negatif.")
        df.loc[df['duration_seconds'] < 0, 'duration_seconds'] = 0
    
    # Feature engineering: Create more robust features
    
    # 1. Cyclical time features instead of raw hour/day
    df['hour_sin'] = np.sin(2 * np.pi * df['start_time'].dt.hour / 24)
    df['hour_cos'] = np.cos(2 * np.pi * df['start_time'].dt.hour / 24)
    df['day_sin'] = np.sin(2 * np.pi * df['start_time'].dt.dayofweek / 7)
    df['day_cos'] = np.cos(2 * np.pi * df['start_time'].dt.dayofweek / 7)
    
    # 2. Normalized duration (to account for different flow sampling periods)
    if 'duration_seconds' in df.columns and 'packet_count' in df.columns:
        df['packets_per_second'] = df['packet_count'] / df['duration_seconds'].replace(0, 1)
    
    # 3. Traffic direction ratios
    if 'incoming_packets' in df.columns and 'outgoing_packets' in df.columns:
        total_packets = df['incoming_packets'] + df['outgoing_packets']
        df['incoming_ratio'] = df['incoming_packets'] / total_packets.replace(0, 1)
        df['outgoing_ratio'] = df['outgoing_packets'] / total_packets.replace(0, 1)
    
    if 'incoming_bytes' in df.columns and 'outgoing_bytes' in df.columns:
        total_bytes = df['incoming_bytes'] + df['outgoing_bytes']
        df['incoming_bytes_ratio'] = df['incoming_bytes'] / total_bytes.replace(0, 1)
        df['outgoing_bytes_ratio'] = df['outgoing_bytes'] / total_bytes.replace(0, 1)
    
    # 4. Average packet size
    if 'total_bytes' in df.columns and 'packet_count' in df.columns:
        df['avg_packet_bytes'] = df['total_bytes'] / df['packet_count'].replace(0, 1)
    
    # 5. Protocol concentration: what percentage of the total traffic does each protocol represent?
    if is_training and encoders is None:
        encoders = CategoricalEncoder()
    
    # Handle categorical variables with consistent encoding
    categorical_columns = ['protocol', 'application_protocol', 'direction']
    for col in categorical_columns:
        if col in df.columns:
            if is_training:
                # During training, fit and transform
                df[f'{col}_encoded'] = encoders.fit_transform(df, col)
            else:
                # During testing, only transform
                df[f'{col}_encoded'] = encoders.transform(df, col)
    
    # Store or use reference statistics for better normalization
    computed_reference_stats = None
    numeric_columns = df.select_dtypes(include=['float64', 'int64']).columns.tolist()
    
    # During training phase, calculate and store reference statistics
    if is_training:
        computed_reference_stats = {}
        for col in numeric_columns:
            # Skip columns that we've just created through encoding
            if col.endswith('_encoded'):
                continue
                
            computed_reference_stats[col] = {
                'mean': float(df[col].mean()),
                'std': float(df[col].std()),
                'min': float(df[col].min()),
                'max': float(df[col].max()),
                'median': float(df[col].median()),
                'q25': float(df[col].quantile(0.25)),
                'q75': float(df[col].quantile(0.75))
            }
        print(f"Calculated reference statistics for {len(computed_reference_stats)} numeric features")
    
    # During testing phase, use reference statistics to create normalized features
    if not is_training and reference_stats is not None:
        for col in numeric_columns:
            # Skip columns that were created through encoding
            if col.endswith('_encoded'):
                continue
                
            if col in reference_stats:
                # Z-score normalization using training user's statistics
                if reference_stats[col]['std'] > 0:
                    z_score = (df[col] - reference_stats[col]['mean']) / reference_stats[col]['std']
                    # Create a new column with the normalized values but keep original
                    df[f'{col}_zscore'] = z_score
                
                # Min-max normalization using training user's statistics
                range_val = reference_stats[col]['max'] - reference_stats[col]['min']
                if range_val > 0:
                    min_max = (df[col] - reference_stats[col]['min']) / range_val
                    df[f'{col}_minmax'] = min_max
                
                # Robust scaling using quantiles from training user's statistics
                iqr = reference_stats[col]['q75'] - reference_stats[col]['q25']
                if iqr > 0:
                    robust = (df[col] - reference_stats[col]['median']) / iqr
                    df[f'{col}_robust'] = robust
    
    # Remove original hour and day_of_week columns to prevent the model from using them directly
    if 'hour' in df.columns:
        df.drop('hour', axis=1, inplace=True)
    if 'day_of_week' in df.columns:
        df.drop('day_of_week', axis=1, inplace=True)
    
    # Keep backward compatibility - only return reference_stats if it was passed in
    if reference_stats is not None:
        return df, encoders, computed_reference_stats
    else:
        return df, encoders


# Feature group definitions
def define_feature_groups():
    time_features = [
        'hour_sin', 'hour_cos', 'day_sin', 'day_cos', 'duration_seconds'
    ]
    
    volume_features = [
        'total_bytes', 'bytes_per_second', 'packet_count', 'packets_per_second', 
        'avg_packet_size', 'min_packet_size', 'max_packet_size'
    ]
    
    direction_features = [
        'incoming_ratio', 'outgoing_ratio', 'incoming_bytes_ratio', 
        'outgoing_bytes_ratio', 'direction_encoded'
    ]
    
    # Create mappings for convenience
    feature_groups = {
        'temporal': time_features,
        'volume': volume_features,
        'direction': direction_features
    }
    
    return feature_groups


# Feature selection function
def select_features_by_group(df, group_name=None):
    """
    Select features from a specific group or all available features
    
    Parameters:
    -----------
    df : pandas.DataFrame
        Input DataFrame
    group_name : str or None
        Name of the feature group to select ('temporal', 'volume', 'direction') or None for all
        
    Returns:
    --------
    selected_features : list
        List of selected feature names
    """
    # Get feature group definitions
    feature_groups = define_feature_groups()
    
    # If no group specified, return all available features
    if group_name is None:
        all_features = []
        for features in feature_groups.values():
            all_features.extend(features)
        available_features = [f for f in all_features if f in df.columns]
        print(f"Selected {len(available_features)} features across all groups")
        return available_features
    
    # Select features from the specified group
    if group_name not in feature_groups:
        raise ValueError(f"Unknown feature group: {group_name}. Available groups: {list(feature_groups.keys())}")
    
    group_features = feature_groups[group_name]
    available_features = [f for f in group_features if f in df.columns]
    
    print(f"Selected {len(available_features)} features from {group_name} group: {available_features}")
    return available_features

In [46]:
def create_group_autoencoder(seq_length, n_features, group_name):
    """
    Create a specialized autoencoder with improved architecture
    """
    # Increase model capacity for better pattern capture
    if group_name == 'temporal':
        encoder_units = [48, 32]  # Larger than previous [24, 16]
        bottleneck_units = 16     # Larger than previous 8
        dropout_rate = 0.2
    elif group_name == 'volume':
        encoder_units = [44, 28]  # Larger than previous [20, 12]
        bottleneck_units = 12     # Larger than previous 6
        dropout_rate = 0.25
    elif group_name == 'direction':
        encoder_units = [40, 24]  # Larger than previous [16, 8]
        bottleneck_units = 10      # Larger than previous 4
        dropout_rate = 0.3
    else:
        encoder_units = [48, 32]
        bottleneck_units = 16
        dropout_rate = 0.25
    
    # Input layer with noise for better generalization
    inputs = Input(shape=(seq_length, n_features))
    noise_level = 0.02  # Slightly increased noise
    noise_layer = GaussianNoise(noise_level)(inputs)
    
    # Encoder with increased regularization
    encoded = LSTM(encoder_units[0], activation='relu', 
                  return_sequences=True,
                  kernel_regularizer=l2(0.0005))(noise_layer)  # Increased from 0.001
    encoded = Dropout(dropout_rate)(encoded)
    encoded = LSTM(encoder_units[1], activation='relu', 
                  kernel_regularizer=l2(0.0005))(encoded)
    encoded = Dropout(dropout_rate)(encoded)
    
    # Bottleneck with non-linear activation
    bottleneck = Dense(bottleneck_units, activation='elu',  # Changed from tanh to elu
                      kernel_regularizer=l2(0.0005))(encoded)
    
    # Decoder with skip connections (residual-like structure)
    decoded = RepeatVector(seq_length)(bottleneck)
    decoded = LSTM(encoder_units[1], activation='relu', 
                  return_sequences=True,
                  kernel_regularizer=l2(0.0005))(decoded)
    decoded = Dropout(dropout_rate)(decoded)
    decoded = LSTM(encoder_units[0], activation='relu', 
                  return_sequences=True,
                  kernel_regularizer=l2(0.0005))(decoded)
    
    # Output layer
    outputs = TimeDistributed(Dense(n_features))(decoded)
    
    # Create model with Huber loss (more robust than MAE)
    model = Model(inputs, outputs)
    model.compile(
        optimizer=Adam(learning_rate=0.001),
        loss='mse'  # Huber yerine MSE
    )
    
    print(f"Created enhanced {group_name} autoencoder with {n_features} features")
    model.summary()
    
    return model

# Sequence creation with improved handling of short sequences
def create_sequences(data, seq_length=5):
    """
    Create sequences from time series data
    
    Parameters:
    -----------
    data : numpy.ndarray
        Input data array
    seq_length : int
        Length of each sequence
    
    Returns:
    --------
    sequences : numpy.ndarray
        Array of sequences
    """
    # Handle case where data is shorter than sequence length
    if len(data) < seq_length:
        # Create padding with zeros
        padding = np.zeros((seq_length - len(data), data.shape[1]))
        data = np.vstack([padding, data])
        
        # Return a single sequence (the padded data)
        return np.array([data])
    
    # Normal case: create sliding window sequences
    sequences = []
    for i in range(len(data) - seq_length + 1):
        seq = data[i:i + seq_length]
        sequences.append(seq)
    
    return np.array(sequences)

In [47]:
def calculate_advanced_deviation(X_test_seq, reconstructed, features, train_error_mean, group_name=None):
    """Calculate deviations with improved polynomial scaling"""
    n_samples = X_test_seq.shape[0]
    
    # Increased scale factors to amplify differences
    if group_name == 'temporal':
        scale_factor = 35.0  # Increased from 15.0
    elif group_name == 'volume':
        scale_factor = 40.0  # Increased from 20.0
    elif group_name == 'direction':
        scale_factor = 45.0  # Increased from 18.0
    else:
        scale_factor = 35.0
    
    # Polynomial power factor (greater than 1 increases contrast)
    power = 1.5
    
    # Calculate errors for each sample
    sample_deviations = np.zeros(n_samples)
    feature_deviations = {feature: 0.0 for feature in features}
    
    for i in range(n_samples):
        # Calculate error for each feature
        for j, feature in enumerate(features):
            error = np.mean(np.abs(X_test_seq[i, :, j] - reconstructed[i, :, j]))
            normalized_error = error / train_error_mean if train_error_mean > 0 else 0
            
            # Polynomial scaling instead of logarithmic
            feature_deviation = min(100, scale_factor * (normalized_error ** power))
                
            # Add to feature totals for averages
            feature_deviations[feature] += feature_deviation
            
            # Add contribution to sample deviation
            sample_deviations[i] += feature_deviation
    
    # Average sample deviations across features
    sample_deviations /= len(features)
    
    # Average feature deviations across samples
    for feature in features:
        feature_deviations[feature] /= n_samples
    
    return sample_deviations, feature_deviations

def calculate_dynamic_threshold(deviations, distance_ratio, group_name=None):
    """Calculate adaptive threshold with improved distance scaling"""
    percentiles = {p: np.percentile(deviations, p) for p in [50, 75, 90, 95, 99]}
    
    # Use more aggressive percentile selection
    if group_name == 'temporal':
        base_percentile = 85 if distance_ratio < 3.0 else 70
    elif group_name == 'volume':
        base_percentile = 90 if distance_ratio < 3.0 else 80
    elif group_name == 'direction':
        base_percentile = 95 if distance_ratio < 3.0 else 85
    else:
        if distance_ratio < 1.5:
            base_percentile = 95
        elif distance_ratio < 2.5:
            base_percentile = 90
        elif distance_ratio < 4.0:
            base_percentile = 80
        else:
            base_percentile = 70
    
    # Use the nearest available percentile
    nearest_percentile = min([50, 75, 90, 95, 99], key=lambda p: abs(p - base_percentile))
    threshold = percentiles[nearest_percentile]
    
    # More aggressive threshold scaling with exponential component
    threshold = min(threshold * (1 + 0.2 * distance_ratio + 0.05 * (distance_ratio ** 2)), 90)
    
    print(f"Percentiles: 50%={percentiles[50]:.2f}, 75%={percentiles[75]:.2f}, 90%={percentiles[90]:.2f}")
    print(f"Selected percentile: {nearest_percentile}, Threshold: {threshold:.2f}")
    
    return threshold

In [48]:
def train_group_autoencoder(train_dirs, group_name, output_dir='model_output', seq_length=5):
    """
    Train an autoencoder for a specific feature group
    
    Parameters:
    -----------
    train_dirs : list
        List of directories containing training data
    group_name : str
        Feature group to train on ('temporal', 'volume', 'direction')
    output_dir : str
        Base directory to save model outputs
    seq_length : int
        Sequence length for time series analysis
        
    Returns:
    --------
    model_info : dict
        Dictionary containing model information
    """
    # Create group-specific output directory
    group_output_dir = os.path.join(output_dir, f"{group_name}_model")
    os.makedirs(group_output_dir, exist_ok=True)
    
    # Load and combine training data
    all_flows = []
    for train_dir in train_dirs:
        flows = load_flows_from_directory(train_dir)
        all_flows.extend(flows)
    
    print(f"Loaded {len(all_flows)} total flows for {group_name} model training")
    
    # Prepare data with encoders
    encoders = CategoricalEncoder()
    train_df, encoders = prepare_flow_data(all_flows, encoders, is_training=True)
    
    # Save encoders
    encoders.save(os.path.join(group_output_dir, 'categorical_encoders.pkl'))
    
    # Select features from the specified group
    features = select_features_by_group(train_df, group_name)
    
    if len(features) == 0:
        raise ValueError(f"No features found for group: {group_name}")
    
    # Scale features
    scaler = MinMaxScaler(feature_range=(-1, 1))
    X = train_df[features].values
    X_scaled = scaler.fit_transform(X)
    
    # Save scaler
    with open(os.path.join(group_output_dir, 'feature_scaler.pkl'), 'wb') as f:
        pickle.dump(scaler, f)
    
    # Create sequences
    X_sequences = create_sequences(X_scaled, seq_length)
    print(f"Created {X_sequences.shape[0]} sequences for {group_name} model training")
    
    # Split into training and validation
    X_train, X_val = train_test_split(X_sequences, test_size=0.2, random_state=42)
    
    # Create group-specific model
    model = create_group_autoencoder(seq_length, len(features), group_name)
    
    # Set up callbacks
    callbacks = [
        EarlyStopping(
            monitor='val_loss',
            patience=10,
            restore_best_weights=True,
            verbose=1
        ),
        ReduceLROnPlateau(
            monitor='val_loss',
            factor=0.2,
            patience=3,
            min_lr=0.00001,
            verbose=1
        ),
        ModelCheckpoint(
            os.path.join(group_output_dir, 'best_model.h5'),
            monitor='val_loss',
            save_best_only=True,
            verbose=1
        )
    ]
    
    # Train the model
    history = model.fit(
        X_train, X_train,
        epochs=100,
        batch_size=16,
        validation_data=(X_val, X_val),
        callbacks=callbacks,
        verbose=1
    )
    
    # Save the final model
    model.save(os.path.join(group_output_dir, 'final_model.h5'))
    
    # Calculate reconstruction errors
    train_pred = model.predict(X_train)
    val_pred = model.predict(X_val)
    
    train_errors = np.mean(np.abs(X_train - train_pred), axis=(1, 2))
    val_errors = np.mean(np.abs(X_val - val_pred), axis=(1, 2))
    
    # Determine threshold (95th percentile)
    threshold = np.percentile(train_errors, 95)
    
    # Visualize training results
    plt.figure(figsize=(15, 10))
    
    # Loss curves
    plt.subplot(2, 2, 1)
    plt.plot(history.history['loss'], label='Training Loss')
    plt.plot(history.history['val_loss'], label='Validation Loss')
    plt.title(f'{group_name.capitalize()} Model Loss')
    plt.xlabel('Epoch')
    plt.ylabel('Loss (MAE)')
    plt.legend()
    plt.grid(True, alpha=0.3)
    
    # Error distribution
    plt.subplot(2, 2, 2)
    plt.hist(train_errors, bins=50, alpha=0.5, label='Training')
    plt.hist(val_errors, bins=50, alpha=0.5, label='Validation')
    plt.axvline(threshold, color='r', linestyle='--', 
               label=f'Threshold ({threshold:.4f})')
    plt.title(f'{group_name.capitalize()} Reconstruction Error Distribution')
    plt.xlabel('Mean Absolute Error (MAE)')
    plt.ylabel('Frequency')
    plt.legend()
    plt.grid(True, alpha=0.3)
    
    # Save visualization
    plt.tight_layout()
    plt.savefig(os.path.join(group_output_dir, 'training_summary.png'))
    plt.close()
    
    # Save model information
    model_info = {
        'group_name': group_name,
        'features': features,
        'seq_length': seq_length,
        'threshold': float(threshold),
        'train_error_mean': float(np.mean(train_errors)),
        'train_error_std': float(np.std(train_errors)),
        'train_error_percentiles': {
            p: float(np.percentile(train_errors, p))
            for p in range(0, 101, 5)
        },
        'date_trained': datetime.now().strftime('%Y-%m-%d %H:%M:%S')
    }
    
    with open(os.path.join(group_output_dir, 'model_info.json'), 'w') as f:
        json.dump(model_info, f, indent=4)
    
    print(f"{group_name.capitalize()} model training completed and saved to {group_output_dir}")
    
    return model_info

In [49]:
def calculate_statistical_deviation(test_df, reference_df, feature_group):
    """
    Calculate statistical deviation between test and reference data for a specific feature group
    
    Parameters:
    -----------
    test_df : pandas.DataFrame
        DataFrame containing test user's data
    reference_df : pandas.DataFrame
        DataFrame containing reference (training) user's data
    feature_group : str
        Name of the feature group ('temporal', 'volume', 'direction')
        
    Returns:
    --------
    deviation : float
        Statistical deviation percentage (0-100)
    feature_deviations : dict
        Per-feature deviation percentages
    """
    # Select features based on feature group
    feature_groups = define_feature_groups()
    features = feature_groups.get(feature_group, [])
    
    # Ensure features exist in both DataFrames
    valid_features = [f for f in features if f in test_df.columns and f in reference_df.columns]
    
    if not valid_features:
        print(f"No valid features found for group {feature_group}")
        return 0.0, {}
    
    # Calculate deviations for each feature
    feature_deviations = {}
    
    for feature in valid_features:
        # Get statistics for both datasets
        test_mean = test_df[feature].mean()
        ref_mean = reference_df[feature].mean()
        
        # Calculate percent difference
        if abs(test_mean) + abs(ref_mean) > 0:
            percent_diff = abs(test_mean - ref_mean) / ((abs(test_mean) + abs(ref_mean)) / 2) * 100
            # Cap at 100%
            feature_deviations[feature] = min(100.0, percent_diff)
        else:
            feature_deviations[feature] = 0.0
    
    # Calculate average deviation across features
    avg_deviation = sum(feature_deviations.values()) / len(feature_deviations) if feature_deviations else 0.0
    
    return avg_deviation, feature_deviations

In [50]:
def calculate_hybrid_deviation(autoencoder_deviation, statistical_deviation, weight=0.5):
    """
    Combine autoencoder and statistical deviations for a more robust metric
    
    Parameters:
    -----------
    autoencoder_deviation : float
        Deviation calculated by the autoencoder (0-100)
    statistical_deviation : float
        Deviation calculated by statistical methods (0-100)
    weight : float
        Weight for autoencoder deviation (0-1), statistical gets (1-weight)
        
    Returns:
    --------
    hybrid_deviation : float
        Combined deviation score (0-100)
    """
    # Simple weighted average of the two deviation metrics
    hybrid_deviation = (weight * autoencoder_deviation) + ((1 - weight) * statistical_deviation)
    
    # Apply a non-linear adjustment to increase sensitivity when both methods agree
    agreement_factor = 1 + (min(autoencoder_deviation, statistical_deviation) / 
                          max(autoencoder_deviation, statistical_deviation) if 
                          max(autoencoder_deviation, statistical_deviation) > 0 else 0) * 0.2
    
    # Cap at 100%
    return min(100.0, hybrid_deviation * agreement_factor)

In [51]:
def simple_statistical_comparison(test_df, reference_df, group_name):
    """
    Calculate a simple statistical deviation between test data and reference data
    
    Parameters:
    -----------
    test_df : pandas.DataFrame
        Test user's data
    reference_df : pandas.DataFrame
        Reference user's data (typically user_1)
    group_name : str
        Feature group name ('temporal', 'volume', 'direction')
        
    Returns:
    --------
    deviation : float
        Statistical deviation percentage (0-100)
    """
    # Select appropriate features based on the group
    if group_name == 'temporal':
        # Time-based features
        time_features = []
        if 'hour_sin' in test_df.columns and 'hour_sin' in reference_df.columns:
            time_features.extend(['hour_sin', 'hour_cos'])
        if 'day_sin' in test_df.columns and 'day_sin' in reference_df.columns:
            time_features.extend(['day_sin', 'day_cos'])
        if 'duration_seconds' in test_df.columns and 'duration_seconds' in reference_df.columns:
            time_features.append('duration_seconds')
            
        features = time_features
            
    elif group_name == 'volume':
        # Volume-based features
        volume_features = []
        if 'total_bytes' in test_df.columns and 'total_bytes' in reference_df.columns:
            volume_features.append('total_bytes')
        if 'bytes_per_second' in test_df.columns and 'bytes_per_second' in reference_df.columns:
            volume_features.append('bytes_per_second')
        if 'packet_count' in test_df.columns and 'packet_count' in reference_df.columns:
            volume_features.append('packet_count')
        if 'packets_per_second' in test_df.columns and 'packets_per_second' in reference_df.columns:
            volume_features.append('packets_per_second')
        if 'avg_packet_size' in test_df.columns and 'avg_packet_size' in reference_df.columns:
            volume_features.append('avg_packet_size')
            
        features = volume_features
        
    elif group_name == 'direction':
        # Direction-based features
        direction_features = []
        if 'incoming_ratio' in test_df.columns and 'incoming_ratio' in reference_df.columns:
            direction_features.append('incoming_ratio')
        if 'outgoing_ratio' in test_df.columns and 'outgoing_ratio' in reference_df.columns:
            direction_features.append('outgoing_ratio')
        if 'incoming_bytes_ratio' in test_df.columns and 'incoming_bytes_ratio' in reference_df.columns:
            direction_features.append('incoming_bytes_ratio')
        if 'outgoing_bytes_ratio' in test_df.columns and 'outgoing_bytes_ratio' in reference_df.columns:
            direction_features.append('outgoing_bytes_ratio')
            
        features = direction_features
    else:
        # Default: use all numerical features
        features = [col for col in test_df.columns 
                   if col in reference_df.columns and 
                   test_df[col].dtype in ['float64', 'int64']]
    
    # If no valid features, return zero deviation
    if not features:
        return 0.0
    
    # Calculate percentage differences for each feature
    feature_deviations = []
    for feature in features:
        test_mean = test_df[feature].mean()
        ref_mean = reference_df[feature].mean()
        
        # Avoid division by zero
        if abs(test_mean) + abs(ref_mean) > 0:
            percent_diff = abs(test_mean - ref_mean) / ((abs(test_mean) + abs(ref_mean)) / 2) * 100
            # Cap at 100%
            feature_deviations.append(min(100.0, percent_diff))
    
    # Return average deviation
    if feature_deviations:
        # Weight group-specific features differently
        if group_name == 'temporal':
            # Time patterns are particularly important
            return min(100.0, np.mean(feature_deviations) * 1.5)
        elif group_name == 'volume':
            # Volume features can be quite variable
            return min(100.0, np.mean(feature_deviations) * 1.2)
        else:
            return min(100.0, np.mean(feature_deviations))
    else:
        return 0.0

In [52]:
def test_group_autoencoder(model_dir, test_dirs, group_name, output_dir=None):
    """
    Test a group-specific autoencoder model with enhanced deviation calculations
    """
    # Set output directory
    group_model_dir = os.path.join(model_dir, f"{group_name}_model")
    if output_dir is None:
        output_dir = os.path.join(group_model_dir, 'test_results')
    
    os.makedirs(output_dir, exist_ok=True)
    
    # Load model and related files
    model = load_model(os.path.join(group_model_dir, 'final_model.h5'))
    
    with open(os.path.join(group_model_dir, 'model_info.json'), 'r') as f:
        model_info = json.load(f)
    
    features = model_info['features']
    seq_length = model_info['seq_length']
    train_error_mean = model_info['train_error_mean']
    
    with open(os.path.join(group_model_dir, 'feature_scaler.pkl'), 'rb') as f:
        scaler = pickle.load(f)
    
    # Load encoders
    encoders = CategoricalEncoder()
    encoders.load(os.path.join(group_model_dir, 'categorical_encoders.pkl'))
    
    # Process and test each scenario
    results = {}
    
    # Load reference data (user_1) for statistical comparisons
    reference_dir = list(test_dirs.values())[0]  # First directory (user_1)
    reference_flows = load_flows_from_directory(reference_dir)
    reference_data, _ = prepare_flow_data(reference_flows, encoders, is_training=False)
    
    for scenario_name, test_dir in test_dirs.items():
        try:
            # Load test data
            test_flows = load_flows_from_directory(test_dir)
            
            if not test_flows:
                print(f"Warning: No data found for {scenario_name} scenario")
                continue
            
            # Prepare test data
            test_df, _ = prepare_flow_data(test_flows, encoders, is_training=False)
            
            # Print basic statistics
            print(f"\n{scenario_name.upper()} {group_name.upper()} SCENARIO STATISTICS:")
            print(f"Total flow count: {len(test_df)}")
            
            # Ensure all features are available
            missing_features = [f for f in features if f not in test_df.columns]
            if missing_features:
                print(f"Warning: Missing features in test data: {missing_features}")
                # Add missing features with zeros
                for feature in missing_features:
                    test_df[feature] = 0
            
            # Scale features
            X_test = test_df[features].values
            X_test_scaled = scaler.transform(X_test)
            
            # Create sequences
            X_test_seq = create_sequences(X_test_scaled, seq_length)
            
            if len(X_test_seq) == 0:
                print(f"Warning: Not enough data for {scenario_name} scenario")
                continue
            
            # Generate predictions
            test_pred = model.predict(X_test_seq)
            
            # Calculate raw errors
            raw_errors = np.mean(np.abs(X_test_seq - test_pred), axis=(1, 2))
            
            # Calculate user distance ratio
            user_distance_ratio = np.mean(raw_errors) / train_error_mean
            print(f"User distance ratio ({group_name}): {user_distance_ratio:.2f}")
            
            # Calculate enhanced deviations with polynomial scaling
            sample_deviations, feature_deviations = calculate_advanced_deviation(
                X_test_seq, test_pred, features, train_error_mean, group_name=group_name)
            
            # Calculate improved threshold with exponential distance scaling
            threshold = calculate_dynamic_threshold(sample_deviations, user_distance_ratio, group_name=group_name)
            
            # Identify anomalies
            anomalies = sample_deviations > threshold
            
            # Calculate anomaly statistics
            anomaly_count = np.sum(anomalies)
            total_count = len(sample_deviations)
            anomaly_percentage = (anomaly_count / total_count) * 100 if total_count > 0 else 0
            
            # Calculate statistical deviation (if not reference data)
            if scenario_name != 'user_1' and reference_data is not None:
                # Simple statistical comparison for key features
                stat_deviation = simple_statistical_comparison(test_df, reference_data, group_name)
                print(f"Statistical deviation ({group_name}): {stat_deviation:.2f}%")
            else:
                stat_deviation = 0.0
            
            # Store results
            results[scenario_name] = {
                'errors': {
                    'mean': float(np.mean(raw_errors)),
                    'max': float(np.max(raw_errors)),
                    'min': float(np.min(raw_errors)),
                    'std': float(np.std(raw_errors)),
                    'ratio_to_train': float(user_distance_ratio)
                },
                'deviations': {
                    'mean': float(np.mean(sample_deviations)),
                    'max': float(np.max(sample_deviations)),
                    'min': float(np.min(sample_deviations)),
                    'std': float(np.std(sample_deviations)),
                    'statistical': float(stat_deviation)  # Add statistical deviation
                },
                'anomalies': {
                    'threshold': float(threshold),
                    'count': int(anomaly_count),
                    'percentage': float(anomaly_percentage)
                },
                'feature_deviations': {
                    feature: float(deviation)
                    for feature, deviation in feature_deviations.items()
                }
            }
            
            # Print summary
            print(f"\n{scenario_name.upper()} {group_name.upper()} DEVIATION RESULTS:")
            print(f"Average Deviation: {results[scenario_name]['deviations']['mean']:.2f}%")
            print(f"Maximum Deviation: {results[scenario_name]['deviations']['max']:.2f}%")
            print(f"Anomaly Threshold: {threshold:.2f}%")
            print(f"Anomaly Count: {anomaly_count} / {total_count} ({anomaly_percentage:.2f}%)")
            
            # Create visualizations
            plt.figure(figsize=(15, 10))
            
            # Error distribution
            plt.subplot(2, 2, 1)
            plt.hist(raw_errors, bins=50, alpha=0.7)
            plt.axvline(train_error_mean, color='r', linestyle='--', 
                       label=f'Training Avg ({train_error_mean:.6f})')
            plt.xlabel('Mean Absolute Error (MAE)')
            plt.ylabel('Frequency')
            plt.title(f'{scenario_name.upper()} {group_name.upper()} Error Distribution')
            plt.legend()
            plt.grid(True, alpha=0.3)
            
            # Deviation distribution
            plt.subplot(2, 2, 2)
            plt.hist(sample_deviations, bins=50, alpha=0.7)
            plt.axvline(threshold, color='r', linestyle='--', 
                       label=f'Anomaly Threshold ({threshold:.2f}%)')
            plt.xlabel('Deviation (%)')
            plt.ylabel('Frequency')
            plt.title(f'{scenario_name.upper()} {group_name.upper()} Deviation Distribution')
            plt.legend()
            plt.grid(True, alpha=0.3)
            
            # Feature deviation comparison
            plt.subplot(2, 2, 3)
            feature_names = list(feature_deviations.keys())
            feature_values = list(feature_deviations.values())
            sorted_indices = np.argsort(feature_values)[::-1]  # Sort by descending deviation
            
            plt.bar(range(len(feature_names)), 
                   [feature_values[i] for i in sorted_indices],
                   color='skyblue')
            plt.xticks(range(len(feature_names)), 
                      [feature_names[i] for i in sorted_indices],
                      rotation=45, ha='right')
            plt.xlabel('Features')
            plt.ylabel('Deviation (%)')
            plt.title(f'Feature Contributions to Deviation')
            plt.grid(True, alpha=0.3, axis='y')
            
            # Add a statistical vs autoencoder comparison if not reference
            if scenario_name != 'user_1' and stat_deviation > 0:
                plt.subplot(2, 2, 4)
                plt.bar([0, 1], [np.mean(sample_deviations), stat_deviation], 
                       color=['blue', 'orange'])
                plt.xticks([0, 1], ['Autoencoder', 'Statistical'])
                plt.ylabel('Deviation (%)')
                plt.title('Deviation Method Comparison')
                plt.grid(True, alpha=0.3, axis='y')
            
            plt.tight_layout()
            plt.savefig(os.path.join(output_dir, f'{scenario_name}_{group_name}_results.png'))
            plt.close()
            
        except Exception as e:
            print(f"Error ({scenario_name}, {group_name}): {str(e)}")
            import traceback
            traceback.print_exc()
    
    # Save results
    with open(os.path.join(output_dir, f'{group_name}_test_results.json'), 'w') as f:
        json.dump(results, f, indent=4)
    
    print(f"\n{group_name.capitalize()} model test results saved to {output_dir}")
    
    return results

In [53]:
def combine_group_results(model_dir, test_dirs, output_dir=None):
    """
    Combine results from all three group models and calculate overall deviation
    
    Parameters:
    -----------
    model_dir : str
        Base directory containing all group model directories
    test_dirs : dict
        Dictionary mapping scenario names to test data directories
    output_dir : str
        Directory to save combined results
        
    Returns:
    --------
    combined_results : dict
        Dictionary containing combined test results with overall deviation
    """
    # Set output directory
    if output_dir is None:
        output_dir = os.path.join(model_dir, 'combined_results')
    
    os.makedirs(output_dir, exist_ok=True)
    
    # Define groups
    groups = ['temporal', 'volume', 'direction']
    
    # Define group weights (matching the statistical code weights)
    group_weights = {
        'temporal': 1.5,   # Time patterns are important
        'volume': 1.2,     # Data volume features
        'direction': 1.3   # Traffic direction features
    }
    
    # Load results for each group
    group_results = {}
    for group in groups:
        group_result_file = os.path.join(model_dir, f"{group}_model/test_results/{group}_test_results.json")
        if os.path.exists(group_result_file):
            with open(group_result_file, 'r') as f:
                group_results[group] = json.load(f)
                print(f"Loaded {group} test results")
        else:
            print(f"Warning: Results file not found for {group} model")
    
    # Combine results
    combined_results = {}
    scenarios = set()
    for group_result in group_results.values():
        scenarios.update(group_result.keys())
    
    for scenario in scenarios:
        # Initialize combined metrics
        combined_results[scenario] = {
            'group_metrics': {},
            'combined_metrics': {
                'weighted_deviation': 0.0,
                'max_group_deviation': 0.0,
                'overall_deviation': 0.0,  # New overall deviation that matches statistical methodology
                'anomaly_percentages': {},
                'overall_anomaly_percentage': 0.0
            }
        }
        
        # Collect metrics from each group
        total_weight = 0.0
        weighted_deviation_sum = 0.0
        max_deviation = 0.0
        max_group = None
        
        # Store group deviations for weighted calculation
        group_deviations = {}
        
        total_anomalies = 0
        total_samples = 0
        
        for group in groups:
            if group in group_results and scenario in group_results[group]:
                group_data = group_results[group][scenario]
                
                # Store group-specific metrics
                combined_results[scenario]['group_metrics'][group] = {
                    'mean_deviation': group_data['deviations']['mean'],
                    'max_deviation': group_data['deviations']['max'],
                    'anomaly_percentage': group_data['anomalies']['percentage'],
                    'distance_ratio': group_data['errors']['ratio_to_train']
                }
                
                # Store group deviation for overall calculation
                group_deviations[group] = group_data['deviations']['mean']
                
                # Update weighted average (for existing calculation)
                weight = group_weights.get(group, 1.0)
                total_weight += weight
                weighted_deviation_sum += group_data['deviations']['mean'] * weight
                
                # Track maximum deviation
                if group_data['deviations']['mean'] > max_deviation:
                    max_deviation = group_data['deviations']['mean']
                    max_group = group
                
                # Track anomalies
                anomaly_count = group_data['anomalies']['count']
                total_count = 0  # We need to extract the total count
                if 'anomalies' in group_data and 'percentage' in group_data['anomalies'] and group_data['anomalies']['percentage'] > 0:
                    total_count = int(anomaly_count / (group_data['anomalies']['percentage'] / 100))
                else:
                    # Fallback if percentage is 0
                    total_count = group_data.get('total_count', 0)
                
                combined_results[scenario]['combined_metrics']['anomaly_percentages'][group] = group_data['anomalies']['percentage']
                
                total_anomalies += anomaly_count
                if total_samples == 0:  # Use the first valid count
                    total_samples = total_count
        
        # Calculate combined metrics
        if total_weight > 0:
            # Existing weighted deviation
            combined_results[scenario]['combined_metrics']['weighted_deviation'] = weighted_deviation_sum / total_weight
            
            # New overall deviation calculation that matches the statistical approach
            # Calculate group-weighted overall deviation
            overall_deviation = calculate_overall_deviation(group_deviations, group_weights)
            combined_results[scenario]['combined_metrics']['overall_deviation'] = overall_deviation
        
        # Store other metrics
        combined_results[scenario]['combined_metrics']['max_group_deviation'] = max_deviation
        combined_results[scenario]['combined_metrics']['max_deviation_group'] = max_group
        
        if total_samples > 0:
            overall_anomaly_percentage = (total_anomalies / (total_samples * len(groups))) * 100
            combined_results[scenario]['combined_metrics']['overall_anomaly_percentage'] = overall_anomaly_percentage
    
    # Save combined results
    with open(os.path.join(output_dir, 'combined_results.json'), 'w') as f:
        json.dump(combined_results, f, indent=4)
    
    # Create combined visualization with overall deviation
    visualize_combined_results(combined_results, output_dir)
    
    print(f"Combined results saved to {output_dir}")
    return combined_results

def calculate_overall_deviation(group_deviations, group_weights=None):
    """
    Calculate overall deviation from group deviations
    
    Parameters:
    -----------
    group_deviations : dict
        Dictionary of group deviations {'temporal': value, 'volume': value, 'direction': value}
    group_weights : dict or None
        Optional weights for each group
        
    Returns:
    --------
    float
        Overall deviation score
    """
    # Default weights if not provided
    if group_weights is None:
        group_weights = {
            'temporal': 1.5,   # Time patterns are important
            'volume': 1.2,     # Data volume features
            'direction': 1.3   # Traffic direction features
        }
    
    # Calculate weighted average
    weighted_sum = sum(
        deviation * group_weights.get(group, 1.0)
        for group, deviation in group_deviations.items()
    )
    
    total_weight = sum(
        group_weights.get(group, 1.0) 
        for group in group_deviations.keys()
    )
    
    # Avoid division by zero
    if total_weight > 0:
        return weighted_sum / total_weight
    else:
        return 0.0

def visualize_combined_results(combined_results, output_dir):
    """
    Create improved visualizations for combined results including overall deviation
    """
    scenarios = list(combined_results.keys())
    
    # 1. Create group deviation comparison chart
    plt.figure(figsize=(14, 10))
    
    # Group deviations
    groups = ['temporal', 'volume', 'direction']
    group_colors = {'temporal': 'blue', 'volume': 'green', 'direction': 'red'}
    
    x = np.arange(len(scenarios))
    width = 0.2  # Narrower bars to accommodate overall deviation
    
    # Plot individual group deviations
    for i, group in enumerate(groups):
        deviations = []
        for scenario in scenarios:
            if group in combined_results[scenario]['group_metrics']:
                deviations.append(combined_results[scenario]['group_metrics'][group]['mean_deviation'])
            else:
                deviations.append(0)
        
        plt.bar(x + (i - 1.5) * width, deviations, width, label=f'{group.capitalize()}', 
               color=group_colors[group])
    
    # Add overall deviation bars
    overall_deviations = []
    for scenario in scenarios:
        overall_deviations.append(combined_results[scenario]['combined_metrics']['overall_deviation'])
    
    plt.bar(x + 1.5 * width, overall_deviations, width, label='Overall', color='purple')
    
    plt.xlabel('Scenario', fontsize=12)
    plt.ylabel('Mean Deviation (%)', fontsize=12)
    plt.title('Behavioral Dimension Comparison Across Scenarios', fontsize=14)
    plt.xticks(x, scenarios)
    plt.legend()
    plt.grid(True, alpha=0.3)
    plt.tight_layout()
    
    plt.savefig(os.path.join(output_dir, 'group_deviation_comparison.png'))
    plt.close()
    
    # 2. Create radar charts with overall deviation for each scenario
    for scenario in scenarios:
        # Get deviations for each group and overall
        group_deviations = []
        group_names = []
        
        for group in groups:
            if group in combined_results[scenario]['group_metrics']:
                group_deviations.append(combined_results[scenario]['group_metrics'][group]['mean_deviation'])
                group_names.append(group.capitalize())
        
        # Add overall deviation
        overall_dev = combined_results[scenario]['combined_metrics']['overall_deviation']
        group_deviations.append(overall_dev)
        group_names.append('Overall')
        
        if len(group_deviations) > 0:
            # Create radar chart
            fig = plt.figure(figsize=(8, 6))
            ax = fig.add_subplot(111, polar=True)
            
            # Compute angles for each group
            angles = np.linspace(0, 2*np.pi, len(group_names), endpoint=False).tolist()
            angles += angles[:1]  # Close the loop
            
            # Add data
            values = group_deviations + [group_deviations[0]]  # Close the loop
            
            # Plot data
            ax.plot(angles, values, 'o-', linewidth=2)
            ax.fill(angles, values, alpha=0.25)
            
            # Set labels
            ax.set_thetagrids(np.degrees(angles[:-1]), group_names)
            
            # Set radial limits
            ax.set_ylim(0, max(max(group_deviations) * 1.2, 20))  # Set limit to at least 20%
            
            plt.title(f'{scenario.capitalize()} Behavioral Profile', fontsize=14)
            plt.tight_layout()
            
            plt.savefig(os.path.join(output_dir, f'{scenario}_radar_chart.png'))
            plt.close()

In [54]:
def run_feature_group_analysis(train_dirs, test_dirs, output_dir='group_model_output'):
    """
    Run the complete feature group analysis pipeline with overall deviation calculation
    
    Parameters:
    -----------
    train_dirs : list
        List of directories containing training data
    test_dirs : dict
        Dictionary mapping scenario names to test data directories
    output_dir : str
        Directory to save all outputs
        
    Returns:
    --------
    combined_results : dict
        Combined results from all models
    """
    os.makedirs(output_dir, exist_ok=True)
    
    # Train each group model
    group_models = {}
    for group in ['temporal', 'volume', 'direction']:
        print(f"\n{'='*50}")
        print(f"TRAINING {group.upper()} MODEL")
        print(f"{'='*50}")
        
        model_info = train_group_autoencoder(train_dirs, group, output_dir)
        group_models[group] = model_info
    
    # Test each group model
    group_results = {}
    for group in ['temporal', 'volume', 'direction']:
        print(f"\n{'='*50}")
        print(f"TESTING {group.upper()} MODEL")
        print(f"{'='*50}")
        
        results = test_group_autoencoder(output_dir, test_dirs, group)
        group_results[group] = results
    
    # Combine results with overall deviation
    print(f"\n{'='*50}")
    print(f"COMBINING RESULTS WITH OVERALL DEVIATION")
    print(f"{'='*50}")
    
    combined_results = combine_group_results(output_dir, test_dirs)
    
    # Print summarized results with group and overall deviations
    print(f"\n{'='*50}")
    print(f"DEVIATION SUMMARY")
    print(f"{'='*50}")
    
    for scenario in combined_results:
        print(f"\nScenario: {scenario.upper()}")
        print(f"{'Group':<10} {'Deviation':<10}")
        print(f"{'-'*10} {'-'*10}")
        
        for group in ['temporal', 'volume', 'direction']:
            if group in combined_results[scenario]['group_metrics']:
                deviation = combined_results[scenario]['group_metrics'][group]['mean_deviation']
                print(f"{group.capitalize():<10} {deviation:.2f}%")
        
        overall_dev = combined_results[scenario]['combined_metrics']['overall_deviation']
        print(f"{'Overall':<10} {overall_dev:.2f}%")
    
    return combined_results

In [55]:
base_dir = 'data'

# Define your data directories
train_dirs = [os.path.join(base_dir, 'Umit\\flows')]  # Your User_1 data

test_scenarios = {
    'user_1': os.path.join(base_dir, 'Umit\\flows'),  # User_1 (self-test)
    'user_2': os.path.join(base_dir, 'Burak\\flows'),  # User_2
    'user_3': os.path.join(base_dir, 'Dilara\\flows'),  # User_3
    #'user_4': os.path.join(base_dir, 'user_4\\flows')   # User_4
}

# Run the analysis
results = run_feature_group_analysis(train_dirs, test_scenarios, 'feature_group_models_hybrid')


TRAINING TEMPORAL MODEL
data\Umit\flows klasöründe 33 JSON dosyası bulundu.
data\Umit\flows klasöründen toplam 189696 akış yüklendi.
Loaded 189696 total flows for temporal model training
Calculated reference statistics for 23 numeric features
Selected 5 features from temporal group: ['hour_sin', 'hour_cos', 'day_sin', 'day_cos', 'duration_seconds']
Created 189692 sequences for temporal model training
Created enhanced temporal autoencoder with 5 features
Model: "model_6"
_________________________________________________________________
 Layer (type)                Output Shape              Param #   
 input_7 (InputLayer)        [(None, 5, 5)]            0         
                                                                 
 gaussian_noise_6 (GaussianN  (None, 5, 5)             0         
 oise)                                                           
                                                                 
 lstm_24 (LSTM)              (None, 5, 48)             10368  